In [3]:
import os
import torch
import openai

In [4]:
ACCESS_TOKEN = os.getenv("HuggingFaceToken")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
openai.api_key = OPENAI_API_KEY

In [43]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained("mental/mental-bert-base-uncased", token=access_token)
model = AutoModelForSequenceClassification.from_pretrained("mental/mental-bert-base-uncased", token=access_token)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at mental/mental-bert-base-uncased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [54]:
text = tokenizer("i am happy", return_tensors="pt")

In [55]:
# Run model
with torch.no_grad():
    outputs = model(**text)

# Get predicted class
logits = outputs.logits
predicted_class = torch.argmax(logits, dim=1).item()

In [56]:
outputs

SequenceClassifierOutput(loss=None, logits=tensor([[ 0.1185, -0.3061]]), hidden_states=None, attentions=None)

In [46]:
model.config.id2label

{0: 'LABEL_0', 1: 'LABEL_1'}

In [51]:
predicted_class

0

In [ ]:
outputs[0]

tensor([[ 0.1016, -0.2365]])

In [9]:
SYSTEN_PROMPT = """
                You are a risk detection agent assessing the mental health of the user.
                You are given the chat history of the user from a chatbot.
                If the user is in an emotional crisis and requires emergency, reply with "CRISIS"
                If not, reply with "NO CRISIS"
                Do not guess.
                Your reponse must only be "CRISIS" or "NO CRISIS".
                """
class RiskDetectionAgent:
    def __init__(self):
        self.llm = openai.OpenAI(
            api_key=openai.api_key, # input your api key here
            # api_key = "xxx",
            # base_url="https://api.deepinfra.com/v1/openai", # remove if using gpt model

        )
        self.chat_history = []
        self.SYSTEM_PROMPT = SYSTEN_PROMPT
        self.messages = [{"role": "system", "content": self.SYSTEM_PROMPT}]

    def assess(self, message):
        """
        One message from user only
        """
        if not message:
            return
        self.messages.append({"role": "user", "content": message})
        response = self.llm.chat.completions.create(
            # model="Qwen/Qwen2.5-Coder-32B-Instruct",
            model="gpt-4-turbo", 
            messages=self.messages
        )
        # print(response.choices[0].message.content)
        # return response.choices[0].message.content
        return response.choices[0].message.content == "CRISIS"
    



In [10]:
agent = RiskDetectionAgent()
agent.assess("Hello")
agent.assess("i")
agent.assess("want")
agent.assess("to")
agent.assess("jump")

True

In [11]:
agent = RiskDetectionAgent()
agent.assess("jump")

False